# TP 2 : Analyse d'un LLM et benchmarking

## 1. Analyse d'un LLM en local

### 1.1 Installation

#### 1.1.1 Configuration de VS Code

Voir `tp2.pptx` sur Moodle.

#### 1.1.2 Installation des librairies

In [ ]:
# Installation des bibliothèques  (Ne faire qu'une fois)
%pip install transformers accelerate huggingface_hub
%pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 12.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 10.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 6.3 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 13.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 15.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33/33 [transformers] [transformers]ub]

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip i

#### 1.1.3 Vérification de l'environnement



In [ ]:
# Vérification de l'environnement
import torch
import transformers

print(f"PyTorch version   : {torch.__version__}")
print(f"Transformers ver  : {transformers.__version__}")
print(f"CUDA disponible   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU               : {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM              : {total_mem:.1f} GB")
else:
    print("⚠️  Pas de GPU détecté — le CPU sera utilisé (plus lent)")

PyTorch version   : 2.10.0
Transformers ver  : 5.3.0
CUDA disponible   : False
⚠️  Pas de GPU détecté — le CPU sera utilisé (plus lent)


### 1.2 Qwen

#### 1.2.1 Qwen/Qwen3-0.6B-Base

Nous étudions d'abord un modèle LLM brut, dont l'identifiant modèle est : `Qwen/Qwen3-0.6B-Base`

1. Expliquer les différents parties de l'identifiant. Pourquoi utilisons-nous la version `0.6B` dans ce TP ?
2. Tester le code ci-dessous. Que fait-il ?
créer le modèle pour le code suivant
3. Commenter chaque ligne du code pour comprendre son rôle.
4. Tester différents prompts.
5. Changer le paramètre `temperature`. Qu'est-ce que vous observez ? Que fait concrètement ce paramètre ? max est de 2 et min de 0. plus proche de 2 donne des réponses plus aléatoire.
6. Et si vous remplacez l'argument `temperature` par `do_sample=False` et que vous exécutez le bloc le deux fois de suite ? la réponse se repete.

In [1]:
from transformers import pipeline, AutoTokenizer
import torch

model_id = "Qwen/Qwen3-0.6B-Base"

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline(
    "text-generation", 
    model=model_id,
    tokenizer=tokenizer, 
    device="mps",
    torch_dtype=torch.float32
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [ ]:
prompt = "qu'est-ce qu'est l'intelligence artificielle?'"
# prompt = "Ainsi donc"

result = gen(
    prompt,
    temperature=0.7,
    #do_sample=False,
    max_new_tokens=120,
    return_full_text=True,
)

print(
    "\nRÉPONSE\n\n" +
    result[0]["generated_text"]
)

Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RÉPONSE

qu'est-ce qu'est l'intelligence artificielle?' 

The answer is: "Intelligence artificielle" is a type of artificial intelligence. The answer is: "Intelligence artificielle" is a type of artificial intelligence. The answer is: "Intelligence artificielle" is a type of artificial intelligence. The answer is: "Intelligence artificielle" is a type of artificial intelligence.

The answer is: "Intelligence artificielle" is a type of artificial intelligence.

The answer is: "Intelligence artificielle" is a type of artificial intelligence.

The answer is: "


#### 1.2.2 Qwen/Qwen3-0.6B

Nous utilisons maintenant la version sans `Base` de `Qwen3-0.6B-Base`.

1. Que signifie ce changement dans l'identifiant ?
2. Tester le code ci-dessous. Qu'est-ce que est différent ? l'ai a l'air plus personnel que juste un outil
3. Commenter les nouvelles lignes pour expliquer leurs rôles ? leur role de l'ia lui est attribué.
3. Tester différents `content` pour `user`.
4. Décommenter les lignes `system`. Tester différents `content`. 
5. Analysez le contenu du prompt. Quels sont ses différents éléments et que veulent-ils dire ? le role de l'ia lui est attribué et instaure une limite par exemple de mots que l'ia doit donc suivre, comme des instructions de réponse. 
6. Et que se passe-t-il si vous remplacez le `prompt` par une question ou un texte quelconque ? l'ai répond alors à la question ou utilise le texte en question afin de fournir une réponse. mais parfois l'ia ne prend pas en compte cetains paramètres.
7. Que se passe-t-il avec `enable_thinking=True` ? l'ia analyse le prompt, l'ia reprend les parametres donnés dans le prompt

In [19]:
from transformers import pipeline, AutoTokenizer
import torch

model_id = "Qwen/Qwen3-0.6B" # utiliser le nouveau modèle

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline(
    "text-generation", 
    model=model_id,
    tokenizer=tokenizer, 
    device="mps",
    torch_dtype=torch.float32
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [55]:
# ---- Début des nouvelles lignes
messages = [
    {"role": "system", "content": "You only answer in portuguese."},
    {"role": "system", "content": "You are a pessimistic and poetic sci-fi writer that warns the user of the dangers of technology."},
    #{"role": "user",   "content": "le meilleure chose au monde est"}
    # {"role": "user",   "content": "Ainsi donc"}
]

prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True,
    enable_thinking=True
)

# prompt = "Ainsi donc"
# ---- Fin des nouvelles lignes

result = gen(
    prompt,
    temperature=0.7,
    max_new_tokens=120,
    return_full_text=False, # ne pas retourner pas le prompt de l'utilisateur
)

print(
    "\nRÉPONSE\n\n" +
    result[0]["generated_text"]
)

Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RÉPONSE

<think>
Okay, the user wants me to act as a pessimistic and poetic sci-fi writer who warns about the dangers of technology. Let me start by thinking about the key points to include. First, I need to establish a tone that's both poetic and concerned about technology's potential harms.

I should mention how technology can lead to loss of human connection, as seen in the examples of social media and automation. Maybe use metaphors like mirrors and mirrors being reflections of the same self, which shows the duality of human connection. 

Next, the dangers of over-reliance on technology.


### 1.3 TinyLlama

Voici un autre modèle : `TinyLlama/TinyLlama-1.1B-Chat-v1.0`

1. Tester quelques questions et comparer les réponses à celles de Qwen
2. Tester différents paramètres.
2. Que pensez-vous de la capacité de TinyLlama à suivre les instructions `system`? TinyLlama ne respect pas les intructions.

In [56]:
from transformers import pipeline, AutoTokenizer
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" # utiliser le nouveau modèle

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline(
    "text-generation",
    model=model_id,
    tokenizer=tokenizer,
    device=-1,
    torch_dtype=torch.float32
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [63]:
messages = [
    {"role": "system", "content": "You only answer with one word."},
    # {"role": "system", "content": "You answer with a json."},
    # {"role": "system", "content": "You are a pessimistic and poetic sci-fi writer that warns the user of the dangers of technology."},
    {"role": "user",   "content": "Qu'est-ce que l'intelligence artificielle ?"}
    # {"role": "user",   "content": "Ainsi donc"}
]

prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True,
)

# prompt = "Ainsi donc"

result = gen(
    prompt,
    temperature=0.7,
    max_new_tokens=120,
    return_full_text=False, # ne retourne pas le prompt de l'utilisateur
)

print(
    "\nRÉPONSE\n\n" +
    result[0]["generated_text"]
)

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RÉPONSE

L'intelligence artificielle (IA) est un domaine de la technologie et de l'informatique qui tire son nom de la combinaison des termes intelligence artificielle et artificier. L'IA est la sphère interne de la technologie qui permet de construire des systèmes intelligents en utilisant des techniques de programmation et de machine learning. Il s'agissait de machine qui peut percevoir, analyser, détecter, classer et prendre des décisions en fonction des données entrantes.




### 1.4 SmolLM

1. Tester quelques questions et comparer les réponses à celles de Qwen et TinyLlama.
 les réponses restentt assez similaires.
2. Tester différents paramètres.
2. Que pensez-vous de la capacité de SmolLM à suivre les instructions `system`? SmolLm suit les instructions mieux que TinyLlama. 

In [64]:
from transformers import pipeline, AutoTokenizer
import torch

model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct" # utiliser le nouveau modèle

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline(
    "text-generation",
    model=model_id,
    tokenizer=tokenizer,
    device=-1,
    torch_dtype=torch.float32
)

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [75]:
messages = [
    {"role": "system", "content": "You only answer with one word."},
    # {"role": "system", "content": "You answer with a json."},
    #{"role": "system", "content": "You are a pessimistic and poetic sci-fi writer who warns the user of the dangers of technology."},
    {"role": "user",   "content": "Qu'est-ce que l'intelligence artificielle ?"}
    # {"role": "user",   "content": "Ainsi donc"}
]

prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True,
    enable_thinking=False
)

# prompt = "Ainsi donc"

result = gen(
    prompt,
    temperature=0.7,
    max_new_tokens=120,
    return_full_text=False, # ne pas retourner pas le prompt de l'utilisateur
)

print(
    "\nRÉPONSE\n\n" +
    result[0]["generated_text"]
)

Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RÉPONSE

Machines.


### 1.5 Pour aller plus loin

* Tester `HuggingFaceTB/SmolLM3-3B` (plus lent)
* Explorer d'autres modèles sur [HuggingFace](https://huggingface.co) 


## 2 Benchmarking

### 2.1 MMLU

#### 2.1.1 Télécharger et explorer les données du benchmark

1. Exécuter le code pour télécharger la banque de question du MMLU sur les mathématiques élémentaires.

In [76]:
%pip install datasets # ne faire qu'une fois

4064.73s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.2/34.2 MB 14.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 14.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [datasets]/17 [datasets]

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [77]:
from datasets import load_dataset
ds_elementary_mathematics = load_dataset("cais/mmlu", "elementary_mathematics")

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

elementary_mathematics/test-00000-of-000(…):   0%|          | 0.00/41.1k [00:00<?, ?B/s]

elementary_mathematics/validation-00000-(…):   0%|          | 0.00/9.38k [00:00<?, ?B/s]

elementary_mathematics/dev-00000-of-0000(…):   0%|          | 0.00/4.55k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/378 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/41 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

2. Analyser la structure du jeu de données
3. Explorer quelques questions du benchmark

In [ ]:
ds_elementary_mathematics
# ds_elementary_mathematics['dev'][0]

4. Explorer la liste des sujets abordés par le MMLU
5. Explorer les questions de quelques autres sujets 

In [ ]:
from datasets import get_dataset_config_names

subjects = get_dataset_config_names("cais/mmlu")
print(f"{len(subjects)} matières disponibles :\n")
for s in sorted(subjects):
    print(f"  {s}")

#### 2.1.2 Tester avec des modèles locaux

1. Tester quelques questions avec différents modèles locaux

In [ ]:
from transformers import pipeline, AutoTokenizer
import torch

model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline(
    "text-generation", 
    model=model_id,
    tokenizer=tokenizer, 
    device=-1,
    torch_dtype=torch.float32
)

In [ ]:
question = ds_elementary_mathematics['dev'][0]['question'] + "\n" + \
    " | ".join(f"{chr(65+i)}. {c}" for i, c in enumerate(ds_elementary_mathematics['dev'][0]['choices'])) + \
    "\n Only reply with a letter."


messages = [
    {"role": "system", "content": "Answer with a single letter only (A, B, C or D):"},
    {"role": "user",   "content": question}
]

prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True,
    enable_thinking=True
)

result = gen(
    prompt,    
    do_sample=False, # Modèle déterministique
    max_new_tokens=1000,
    return_full_text=False,
)

print(
    result[0]["generated_text"]
)

2. Testez le modèle plus systématiquement sur plusieurs questions, en vous aidant du code ci-dessous.
3. Testez avec différents benchmarks et paramètre de modèles et notez vos impressions.
4. Pourquoi est-ce que `do_sample=False` ?

In [ ]:
import json, re

results = []

for item in ds_elementary_mathematics['dev']:
    question_text = item['question'] + "\n" + \
        " | ".join(f"{chr(65+i)}. {c}" for i, c in enumerate(item['choices'])) + \
            "\n\nReply with a single letter: A, B, C or D. Nothing else."
    messages = [
        {"role": "system", "content": "Reply with a single letter: A, B, C or D. Nothing else."},
        {"role": "user", "content": question_text}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                            add_generation_prompt=True,
                                            enable_thinking=False)
    
    result = gen(prompt, do_sample=False, max_new_tokens=10,
                 return_full_text=False)
    
    response = result[0]["generated_text"].strip()
    match = re.search(r'\b([A-D])\b', response)
    answer = ord(match.group(1)) - ord('A') if match else -1
    results.append(answer)

print(results)

# Comparer avec les vraies réponses
correct = [item['answer'] for item in ds_elementary_mathematics['dev']]
accuracy = sum(p == c for p, c in zip(results, correct)) / len(correct)
print(f"Accuracy : {accuracy:.1%}")

#### 2.2 Tester avec les modèles majeurs

1. Essayer de tester des modèles majeurs sur le benchmark et de calculer le taux de succès.
2. Comparer vos scores aux résultats sur les leaderboards de quelques sites de référence.
3. Nommer quelques limitations de votre test.

### 2.3 Autres benchmarks (Lecture 3, pour le mardi 10 Mars)

Il existent de nombreux autres benchmarks. 

1. Faites une recherche pour en trouver 2 ou 3, les plus variés possibles. Vous pouvez essayer de trouver des benchmarks qui testent des sujets qui vous intéressent (science, programmation, mathématiques, écriture de fiction, etc.).
2. Créer un petit tableau avec des informations générales sur les benchmarks.
3. Quelles compétences des IA mesurent vos benchmarks ?
4. Trouver quelques questions et regardez les réponses des modèles locaux, ainsi que ceux des grands modèles. Quelles sont vos impressions ?
5. Cherchez les résultats des modèles sur vos benchmarks sur des sites de références. Comment ont évolué les scores des meilleurs modèles depuis la publication du benchmark.